<a href="https://colab.research.google.com/github/racoope70/daytrading-with-ml/blob/main/multi_ticker_LightGBN_walkforward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Remove conflicting CUDA first
!apt-get remove --purge -y cuda* libcuda* nvidia* || echo "No conflicting CUDA packages"
!apt-get autoremove -y && apt-get clean

#Install compatible RAPIDS + LightGBM
!pip install lightgbm==4.1.0
!pip install --extra-index-url=https://pypi.nvidia.com \
    cupy-cuda12x cudf-cu12==24.4.0 cuml-cu12==24.4.0 dask-cudf-cu12==24.4.0


In [2]:
!pip install pandas numpy yfinance matplotlib scikit-learn lightgbm==4.1.0 joblib

In [1]:
!pip install tensorflow==2.18.0

In [5]:
!pip install yfinance pandas numpy matplotlib scikit-learn lightgbm xgboost joblib

In [6]:
!pip install stable-baselines3 gymnasium gym-anytrading

In [3]:
import os
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")



In [18]:
#Clear saved models and scalers
for fname in os.listdir(save_dir):
    if fname.startswith("lgb_") or fname.endswith("_scaler.pkl"):
        os.remove(os.path.join(save_dir, fname))
        print(f"Deleted: {fname}")

gc.collect()


In [19]:
#Imports
import os, gc, joblib, yfinance as yf
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier, plot_importance
from lightgbm import early_stopping, log_evaluation

#Config
save_dir = '/content/drive/MyDrive/LightGBM_Models'
os.makedirs(save_dir, exist_ok=True)
test_mode = False
force_retrain = True

test_mode = False
TICKERS = ['AAPL'] if test_mode else [
    'AAPL', 'TSLA', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'BRK-B', 'JPM', 'JNJ',
    'XOM', 'V', 'PG', 'UNH', 'MA', 'HD', 'LLY', 'MRK', 'PEP', 'KO',
    'BAC', 'ABBV', 'AVGO', 'PFE', 'COST', 'CSCO', 'TMO', 'ABT', 'ACN', 'WMT',
    'MCD', 'ADBE', 'DHR', 'CRM', 'NKE', 'INTC', 'QCOM', 'NEE', 'AMD', 'TXN',
    'AMGN', 'UPS', 'LIN', 'PM', 'UNP', 'BMY', 'LOW', 'RTX', 'CVX', 'IBM',
    'GE', 'SBUX', 'ORCL'
]


required_cols = ['Close', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'Signal_Line', 'ATR', 'OBV', 'CCI']

def compute_technical_indicators(df):
    try:
        df = df.copy()
        df['SMA_50'] = df['Close'].rolling(window=50).mean()
        df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(window=14).mean()
        loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
        rs = gain / (loss + 1e-6)
        df['RSI'] = 100 - (100 / (1 + rs))
        df['MACD'] = df['Close'].ewm(span=12, adjust=False).mean() - df['Close'].ewm(span=26, adjust=False).mean()
        df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
        df['ATR'] = df['High'].rolling(window=14).max() - df['Low'].rolling(window=14).min()
        df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
        typical_price = (df['High'] + df['Low'] + df['Close']) / 3
        df['CCI'] = (typical_price - typical_price.rolling(20).mean()) / (0.015 * typical_price.rolling(20).std())
        df.dropna(inplace=True)
        return df
    except Exception as e:
        print(f"Error computing indicators: {e}")
        return pd.DataFrame()

def generate_labels(df):
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
    df.dropna(inplace=True)
    return df

def train_lightgbm_for_stock(ticker, force_retrain=False):
    print(f"\nTraining LightGBM for {ticker}")
    model_path = f"{save_dir}/lgb_{ticker}_model.txt"
    scaler_path = f"{save_dir}/{ticker}_scaler.pkl"

    if os.path.exists(model_path) and not test_mode and not force_retrain:
        print(f"⏩ Skipping {ticker}, model already exists.")
        return

    df = yf.download(ticker, period="720d", interval="1h", progress=False, auto_adjust=False)
    if df.empty:
        print(f"No data for {ticker}, skipping.")
        return

    df = compute_technical_indicators(df)

    missing = [col for col in required_cols if col not in df.columns]
    if df.empty or missing:
        print(f"Skipping {ticker} due to error: {missing if missing else 'empty dataframe'}")
        print(df.head())
        return

    df = generate_labels(df)
    if df.empty:
        print(f"Skipping {ticker}: not enough data after labeling.")
        return

    X = df[required_cols]
    y = df['Target']

    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, shuffle=False)

    model = LGBMClassifier(
        objective='binary',
        boosting_type='gbdt',
        learning_rate=0.05,
        n_estimators=500,
        random_state=42
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[early_stopping(stopping_rounds=20), log_evaluation(period=50)]
    )

    model.booster_.save_model(model_path)
    joblib.dump(scaler, scaler_path)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{ticker} model saved | Accuracy: {acc:.4f}")

    if test_mode and ticker == 'AAPL':
        plot_importance(model, max_num_features=10, importance_type='gain')
        plt.title("Top 10 AAPL Feature Importances")
        plt.tight_layout()
        plt.show()

#Run training
for ticker in TICKERS:
    train_lightgbm_for_stock(ticker, force_retrain=force_retrain)
    gc.collect()


AAPL model saved | Accuracy: 0.4909
TSLA model saved | Accuracy: 0.4970
MSFT model saved | Accuracy: 0.5050
GOOGL model saved | Accuracy: 0.5151
AMZN model saved | Accuracy: 0.5131
NVDA model saved | Accuracy: 0.5151
META model saved | Accuracy: 0.5020
BRK-B model saved | Accuracy: 0.5221
JPM model saved | Accuracy: 0.5302
JNJ model saved | Accuracy: 0.4970
XOM model saved | Accuracy: 0.4849
V model saved | Accuracy: 0.4930
PG model saved | Accuracy: 0.4869
UNH model saved | Accuracy: 0.5201
MA model saved | Accuracy: 0.5111
HD model saved | Accuracy: 0.4960
LLY model saved | Accuracy: 0.4960
MRK model saved | Accuracy: 0.5091
PEP model saved | Accuracy: 0.4809
KO model saved | Accuracy: 0.4980
BAC model saved | Accuracy: 0.4849
ABBV model saved | Accuracy: 0.5050
AVGO model saved | Accuracy: 0.5181
PFE model saved | Accuracy: 0.5141
COST model saved | Accuracy: 0.5221
CSCO model saved | Accuracy: 0.5050
TMO model saved | Accuracy: 0.4759
ABT model saved | Accuracy: 0.5080
ACN model sa

In [24]:
#Imports
import os, joblib, yfinance as yf
import pandas as pd, numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt

#Required Features
features = ['Close', 'SMA_50', 'EMA_20', 'RSI', 'MACD', 'Signal_Line', 'ATR', 'OBV', 'CCI']

#Indicator Computation
def compute_technical_indicators(df):
    try:
        df = df.copy()
        df['SMA_50'] = df['Close'].rolling(window=50).mean()
        df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
        delta = df['Close'].diff()
        gain = delta.where(delta > 0, 0).rolling(window=14).mean()
        loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
        rs = gain / (loss + 1e-6)
        df['RSI'] = 100 - (100 / (1 + rs))
        df['MACD'] = df['Close'].ewm(span=12, adjust=False).mean() - df['Close'].ewm(span=26, adjust=False).mean()
        df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()
        df['ATR'] = df['High'].rolling(window=14).max() - df['Low'].rolling(window=14).min()
        df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
        tp = (df['High'] + df['Low'] + df['Close']) / 3
        df['CCI'] = (tp - tp.rolling(20).mean()) / (0.015 * tp.rolling(20).std())

        df.dropna(inplace=True)

        if df.empty or len(df) < 100:
            print(f"Data too short after indicator computation: {len(df)} rows")
            return pd.DataFrame()

        return df
    except Exception as e:
        print(f"Error computing indicators: {e}")
        return pd.DataFrame()

#Walkforward Evaluation
def walkforward_evaluate_lightgbm_model(ticker, model_path, scaler_path,
                                        sequence_length=3000, step_size=500,
                                        initial_cash=100000, return_history=False):
    #Load model and scaler
    model = lgb.Booster(model_file=model_path)
    scaler = joblib.load(scaler_path)

    #Download and process data
    df = yf.download(ticker, period="720d", interval="1h", progress=False)
    df = compute_technical_indicators(df)

    if df.empty or not all(col in df.columns for col in features):
        print(f"Skipping evaluation for {ticker}: Missing required columns.")
        return None

    df_feat = df[features].copy()

    if hasattr(scaler, "feature_names_in_"):
        df_feat = df_feat[scaler.feature_names_in_]

    prices = df["Close"].values
    dates = df.index

    portfolio_lgbm = []
    portfolio_hold = []

    capital = initial_cash
    shares = initial_cash / prices[0]

    for start in range(0, len(df_feat) - sequence_length, step_size):
        end = start + sequence_length
        window = df_feat.iloc[start:end]
        window_prices = prices[start:end]
        X_scaled = scaler.transform(window)
        y_pred = (model.predict(X_scaled) > 0.5).astype(int)

        for i in range(1, len(window_prices)):
            signal = y_pred[i-1]
            price = window_prices[i]
            if signal:
                capital = shares * price
            else:
                shares = capital / price
            portfolio_lgbm.append(capital)
            portfolio_hold.append(shares * price)

    result = {
        "Final Portfolio (LGBM)": portfolio_lgbm[-1],
        "Final Portfolio (Hold)": portfolio_hold[-1],
        "Return % (LGBM)": (portfolio_lgbm[-1] - initial_cash) / initial_cash * 100,
        "Return % (Hold)": (portfolio_hold[-1] - initial_cash) / initial_cash * 100
    }

    if return_history:
        result["Portfolio_LGBM_History"] = portfolio_lgbm
        result["Portfolio_Hold_History"] = portfolio_hold
        result["Dates"] = dates[-len(portfolio_lgbm):].to_list()

    return result


In [25]:
results = []

for ticker in TICKERS:
    try:
        model_path = f"/content/drive/MyDrive/LightGBM_Models/lgb_{ticker}_model.txt"
        scaler_path = f"/content/drive/MyDrive/LightGBM_Models/{ticker}_scaler.pkl"
        if not os.path.exists(model_path) or not os.path.exists(scaler_path):
            print(f" Missing model or scaler for {ticker}, skipping.")
            continue

        result = walkforward_evaluate_lightgbm_model(
            ticker=ticker,
            model_path=model_path,
            scaler_path=scaler_path,
            sequence_length=3000,
            step_size=500,
            initial_cash=100000,
            return_history=False
        )

        result["Symbol"] = ticker
        results.append(result)

    except Exception as e:
        print(f"Skipping {ticker} due to error: {e}")

In [26]:
import pandas as pd

#Ensure better display in console (optional)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

#Create Summary DataFrame
results_df = pd.DataFrame(results)

#Check if results are available
if results_df.empty:
    print("No results to display.")
else:
    #Sort and extract Top/Bottom Performers
    top5 = results_df.sort_values(by="Return % (LGBM)", ascending=False).head(5)
    bottom5 = results_df.sort_values(by="Return % (LGBM)", ascending=True).head(5)

    print("\n Top 5 LGBM Stocks by Return:")
    print(top5.to_string(index=False))

    print("\nBottom 5 LGBM Stocks by Return:")
    print(bottom5.to_string(index=False))



 Top 5 LGBM Stocks by Return:
Final Portfolio (LGBM) Final Portfolio (Hold)      Return % (LGBM)      Return % (Hold) Symbol
  [135543022.93937826]   [135543022.93937826] [135443.02293937825] [135443.02293937825]    UNP
  [121599136.29765114]   [121599136.29765114] [121499.13629765114] [121499.13629765114]    UNH
    [78364753.0106541]     [78364753.0106541]   [78264.7530106541]   [78264.7530106541]    BMY
   [72132117.13479328]    [72132117.13479328]  [72032.11713479328]  [72032.11713479328]    NKE
    [58129421.0696123]     [58129421.0696123] [58029.421069612305] [58029.421069612305]   AMGN

Bottom 5 LGBM Stocks by Return:
Final Portfolio (LGBM) Final Portfolio (Hold)       Return % (LGBM)       Return % (Hold) Symbol
    [89104.9011366634]     [89104.9011366634] [-10.895098863336607] [-10.895098863336607]    NEE
  [108948.88832504956]   [108948.88832504956]   [8.948888325049555]   [8.948888325049555]   ADBE
  [110748.67537892942]   [110748.67537892942]  [10.748675378929416]  [10.74